# Inducción matemática 🪜
### Una introducción práctica (con aplicaciones a Computer Science)

La inducción matemática es, probablemente, **la técnica de demostración más importante que usarás en CS**.
Se usa para probar que un algoritmo recursivo es correcto, que un bucle hace lo que debe, o que una fórmula
funciona para *cualquier* tamaño de entrada, por grande que sea.

**Índice**
1. La idea intuitiva: el efecto dominó
2. Estructura formal de una demostración por inducción
3. Primer ejemplo: la suma de los primeros n números
4. Verificación con Python
5. Inducción fuerte
6. Aplicación: corrección de un algoritmo recursivo
7. Aplicación: invariantes de bucle
8. Aplicación: las Torres de Hanói
9. Errores comunes al usar inducción
10. Ejercicios para practicar

## 1. La idea intuitiva: el efecto dominó

Imagina una fila infinita de fichas de dominó. Quieres demostrar que **todas van a caer**.

Te basta con comprobar dos cosas:
1. **La primera ficha cae.** (caso base)
2. **Si una ficha cualquiera cae, entonces empuja y tira a la siguiente.** (paso inductivo)

Si se cumplen ambas, sabes con certeza que *todas* las fichas van a caer, sin necesidad de empujarlas una por una.

La inducción matemática funciona exactamente igual, pero en vez de fichas de dominó, hablamos de **una afirmación
$P(n)$ que depende de un número natural $n$**, y queremos demostrar que es cierta para todo $n \geq n_0$.

In [ ]:
# Simulación del efecto dominó: si cada ficha empuja a la siguiente,
# comprobamos que todas caen a partir de la primera.

def simular_domino(n_fichas):
    caidas = [False] * n_fichas
    caidas[0] = True  # Caso base: la primera ficha cae
    for i in range(1, n_fichas):
        # Paso inductivo: si la ficha anterior cayó, esta también cae
        if caidas[i - 1]:
            caidas[i] = True
    return caidas

resultado = simular_domino(15)
print('¿Cuántas fichas caen?', sum(resultado), 'de', len(resultado))
print(resultado)

## 2. Estructura formal de una demostración por inducción

Para demostrar que una afirmación $P(n)$ es cierta para todo entero $n \geq n_0$, seguimos tres pasos:

### Paso 1 — Caso base
Demostrar que $P(n_0)$ es cierta (normalmente $n_0 = 0$ o $n_0 = 1$).

### Paso 2 — Hipótesis de inducción
**Suponer** que $P(k)$ es cierta para un $k \geq n_0$ arbitrario (esto NO lo demostramos, lo asumimos como hipótesis).

### Paso 3 — Paso inductivo
Usando esa hipótesis, **demostrar** que $P(k+1)$ también es cierta.

Si completas los tres pasos, $P(n)$ queda demostrada para **todo** $n \geq n_0$, automáticamente, igual que con las fichas de dominó.

## 3. Primer ejemplo: la suma de los primeros $n$ números

Queremos demostrar que para todo $n \geq 1$:

$$P(n): \quad 1 + 2 + 3 + \dots + n = \dfrac{n(n+1)}{2}$$

**Caso base ($n=1$):** el lado izquierdo es $1$, el derecho es $\dfrac{1 \cdot 2}{2} = 1$. ✅ Coinciden.

**Hipótesis de inducción:** suponemos que $P(k)$ es cierta, es decir:
$$1 + 2 + \dots + k = \dfrac{k(k+1)}{2}$$

**Paso inductivo:** queremos ver que $P(k+1)$ también se cumple. Partimos de la suma hasta $k+1$ y
usamos la hipótesis:

$$1 + 2 + \dots + k + (k+1) = \underbrace{\dfrac{k(k+1)}{2}}_{\text{hipótesis}} + (k+1) = \dfrac{k(k+1) + 2(k+1)}{2} = \dfrac{(k+1)(k+2)}{2}$$

Y $\dfrac{(k+1)(k+2)}{2}$ es exactamente la fórmula $P(k+1)$ que queríamos demostrar. **Fin de la demostración.** ∎

## 4. Verificación (no demostración) con Python

Python no puede *demostrar* la fórmula para todo $n$ (eso solo lo hace una prueba matemática, ya que hay infinitos
números), pero sí podemos **comprobarla empíricamente** para muchos valores, lo cual da mucha confianza y
ayuda a entender por qué funciona.

In [ ]:
def suma_directa(n):
    return sum(range(1, n + 1))

def suma_formula(n):
    return n * (n + 1) // 2

todos_coinciden = all(suma_directa(n) == suma_formula(n) for n in range(1, 1000))
print('¿Coinciden para n = 1..999?', todos_coinciden)

for n in [1, 5, 10, 100]:
    print(f'n={n:>3}: suma directa={suma_directa(n):>6}, fórmula={suma_formula(n):>6}')

## 5. Inducción fuerte

A veces, para demostrar $P(k+1)$ no basta con suponer $P(k)$: necesitamos suponer que **todas** las afirmaciones
anteriores $P(n_0), P(n_0+1), \dots, P(k)$ son ciertas. A esto se le llama **inducción fuerte**, y es habitual
cuando trabajamos con recursiones que no dependen solo del caso inmediatamente anterior (como Fibonacci).

**Ejemplo típico en CS: todo número natural $n \geq 2$ se puede escribir como producto de números primos**
(el Teorema Fundamental de la Aritmética). Para descomponer $n$, a veces necesitas descomponer un divisor $d < n$
que no es $n-1$, así que necesitas la hipótesis para *todos* los valores anteriores, no solo el inmediato.

In [ ]:
def factorizacion_prima(n):
    """Ejemplo de razonamiento por inducción fuerte, convertido en algoritmo:
    para factorizar n, buscamos CUALQUIER divisor d < n, no necesariamente n-1."""
    factores = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            factores.append(d)
            n //= d
        d += 1
    if n > 1:
        factores.append(n)
    return factores

for numero in [12, 17, 100, 360]:
    print(f'{numero} = {" x ".join(map(str, factorizacion_prima(numero)))}')

## 6. Aplicación: demostrar que un algoritmo recursivo es correcto

Esta es la aplicación más importante en CS: **la inducción y la recursión son dos caras de la misma moneda**.
Cada función recursiva tiene un caso base y un caso recursivo, ¡igual que una demostración por inducción!

Vamos a demostrar que esta función que calcula la potencia $b^n$ es correcta para todo $n \geq 0$:

```python
def potencia(b, n):
    if n == 0:
        return 1
    return b * potencia(b, n - 1)
```

**Caso base ($n=0$):** `potencia(b, 0)` devuelve `1`, y sabemos que $b^0 = 1$. ✅

**Hipótesis de inducción:** suponemos que `potencia(b, k)` devuelve correctamente $b^k$.

**Paso inductivo:** `potencia(b, k+1)` devuelve `b * potencia(b, k)`, que por la hipótesis es $b \cdot b^k = b^{k+1}$. ✅

Por inducción, `potencia(b, n)` es correcta para todo $n \geq 0$.

In [ ]:
def potencia(b, n):
    if n == 0:
        return 1
    return b * potencia(b, n - 1)

# Comprobación empírica de la demostración anterior
for b in [2, 3, 5]:
    for n in range(0, 10):
        assert potencia(b, n) == b ** n, f'Falla en b={b}, n={n}'
print('Todas las comprobaciones pasaron: potencia(b, n) es correcta ✅')

## 7. Aplicación: invariantes de bucle

La inducción también sirve para demostrar que un **bucle** (no solo una función recursiva) hace lo correcto.
La idea es encontrar un **invariante**: una afirmación que es cierta *antes de empezar*, *se mantiene cierta*
tras cada iteración, y *al terminar el bucle* nos da el resultado que queremos. Es exactamente el mismo
esquema: caso base + paso inductivo.

**Ejemplo:** esta función suma los elementos de una lista con un bucle. El invariante es:

> *Al empezar la iteración `i`, `acumulador` contiene la suma de los primeros `i` elementos.*

In [ ]:
def suma_lista(lista):
    acumulador = 0
    # Invariante ANTES del bucle (caso base, i=0): acumulador = suma de 0 elementos = 0. ✅
    for i, elemento in enumerate(lista):
        # Aquí, por hipótesis de inducción, acumulador = suma de los primeros i elementos
        acumulador += elemento
        # Paso inductivo: ahora acumulador = suma de los primeros i+1 elementos ✅
    # Al salir del bucle, i+1 = len(lista), así que acumulador = suma de TODOS los elementos
    return acumulador

import random
for _ in range(5):
    datos = [random.randint(-50, 50) for _ in range(10)]
    assert suma_lista(datos) == sum(datos)
print('El invariante se mantiene: suma_lista(lista) siempre es correcta ✅')

## 8. Aplicación: las Torres de Hanói

Un clásico. Tenemos 3 postes y $n$ discos de distinto tamaño apilados en el primero (de mayor a menor).
Queremos moverlos todos al tercer poste, moviendo un disco a la vez y **sin poner nunca un disco más grande
encima de uno más pequeño**.

**Afirmación $P(n)$:** el problema con $n$ discos se puede resolver en exactamente $2^n - 1$ movimientos.

**Caso base ($n=1$):** un solo disco se mueve directamente: $2^1 - 1 = 1$ movimiento. ✅

**Hipótesis de inducción:** resolver el problema con $k$ discos cuesta $2^k - 1$ movimientos.

**Paso inductivo:** para mover $k+1$ discos del poste A al C usando B de apoyo:
1. Mover los $k$ discos superiores de A a B (por hipótesis: $2^k - 1$ movimientos),
2. Mover el disco más grande de A a C (1 movimiento),
3. Mover los $k$ discos de B a C (por hipótesis, otra vez: $2^k - 1$ movimientos).

Total: $(2^k - 1) + 1 + (2^k - 1) = 2^{k+1} - 1$. Exactamente $P(k+1)$. ∎

¡Y esta misma demostración **es** el algoritmo recursivo que resuelve el problema!

In [ ]:
def hanoi(n, origen, auxiliar, destino, movimientos=None):
    if movimientos is None:
        movimientos = []
    if n == 1:
        movimientos.append((origen, destino))
        return movimientos
    hanoi(n - 1, origen, destino, auxiliar, movimientos)  # paso 1
    movimientos.append((origen, destino))                 # paso 2
    hanoi(n - 1, auxiliar, origen, destino, movimientos)   # paso 3
    return movimientos

for n in range(1, 8):
    pasos = hanoi(n, 'A', 'B', 'C')
    prediccion = 2**n - 1
    print(f'n={n}: {len(pasos)} movimientos (predicción 2^{n}-1 = {prediccion}) -> {"OK" if len(pasos)==prediccion else "ERROR"}')

print()
print('Solución paso a paso para n=3:')
for origen, destino in hanoi(3, 'A', 'B', 'C'):
    print(f'  Mover disco de {origen} -> {destino}')

In [ ]:
import matplotlib.pyplot as plt

n = 8
movimientos_por_n = [len(hanoi(k, 'A', 'B', 'C')) for k in range(1, n + 1)]

plt.figure(figsize=(7, 4))
plt.plot(range(1, n + 1), movimientos_por_n, 'o-', label='Movimientos reales')
plt.plot(range(1, n + 1), [2**k - 1 for k in range(1, n + 1)], 'x--', label='$2^n - 1$ (predicción)')
plt.xlabel('Número de discos (n)')
plt.ylabel('Movimientos')
plt.title('Torres de Hanói: crecimiento exponencial')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Errores comunes al usar inducción

### ❌ Olvidar el caso base
Sin caso base, no tienes de dónde "arrancar" el dominó. Por ejemplo, se podría "demostrar" (¡mal!) que
todos los caballos son del mismo color con un paso inductivo aparentemente válido, pero **sin verificar
correctamente el caso base para $n=1$ y $n=2$**, la demostración se rompe.

### ❌ El paso inductivo no cubre realmente el caso $k \to k+1$
Hay que revisar con cuidado que el argumento funcione para **cualquier** $k$, no solo para los primeros valores
que se te ocurra probar a mano.

### ❌ Confundir "comprobar para varios valores" con "demostrar para todos"
Que una fórmula funcione para $n = 1, \dots, 1000$ en Python **no es una demostración matemática**: puede fallar
en $n = 1001$. La inducción es lo que garantiza que funciona para *todo* $n$, no la fuerza bruta.

### ❌ Usar inducción simple cuando se necesita inducción fuerte
Si el paso inductivo necesita más de un caso anterior (como en Fibonacci, donde $F(k+1)$ depende de $F(k)$
*y* $F(k-1)$), hay que usar inducción fuerte, no simple.

In [ ]:
# Ejemplo de por qué 'comprobar mucho' no es lo mismo que 'demostrar':
# f(n) = n^2 - n + 41 da un número primo para muchos valores de n... pero no para todos.

def es_primo(x):
    if x < 2:
        return False
    for d in range(2, int(x**0.5) + 1):
        if x % d == 0:
            return False
    return True

def f(n):
    return n**2 - n + 41

primos_seguidos = 0
n = 0
while es_primo(f(n)):
    primos_seguidos += 1
    n += 1

print(f'f(n) = n^2 - n + 41 da un primo para n = 0..{n-1} ({primos_seguidos} valores seguidos)')
print(f'Pero falla en n={n}: f({n}) = {f(n)} = {f(n)} (¿primo? {es_primo(f(n))})')
print('Moraleja: comprobar muchos casos NO es una demostración por inducción.')

## 10. Ejercicios para practicar

Intenta escribir la demostración en la celda de markdown (puedes editar esta celda) y comprueba tu resultado con código.

1. **Demuestra por inducción** que para todo $n \geq 1$: $1 + 3 + 5 + \dots + (2n-1) = n^2$ (la suma de los primeros $n$ números impares).
2. **Demuestra por inducción** que $n! \geq 2^n$ para todo $n \geq 4$ (fíjate en cuál es el caso base correcto).
3. Escribe una función recursiva `suma_impares(n)` que calcule la suma anterior, y compárala con la fórmula $n^2$ para muchos valores.
4. **Desafío:** demuestra por inducción que un árbol binario con $n$ nodos internos tiene exactamente $n+1$ hojas (si cada nodo interno tiene exactamente 2 hijos). Pista: caso base $n=0$ (un solo nodo, que es hoja); en el paso inductivo, añadir un nodo interno siempre convierte 1 hoja en 2.

In [ ]:
# Ejercicio 3: tu código aquí
def suma_impares(n):
    pass  # completa la función recursiva


---
### Solución del ejercicio 3

In [ ]:
def suma_impares(n):
    if n == 1:
        return 1
    return suma_impares(n - 1) + (2 * n - 1)

for n in range(1, 20):
    assert suma_impares(n) == n**2, f'Falla en n={n}'
print('suma_impares(n) coincide con n^2 para n = 1..19 ✅')
print([suma_impares(n) for n in range(1, 10)])

## Resumen

La inducción matemática demuestra que $P(n)$ es cierta para todo $n \geq n_0$ con solo dos ingredientes:

| Paso | Qué haces | Analogía con recursión |
|---|---|---|
| Caso base | Demuestras $P(n_0)$ directamente | Caso base de la función recursiva |
| Paso inductivo | Supones $P(k)$ y demuestras $P(k+1)$ | Llamada recursiva que confía en que el subproblema más pequeño ya funciona |

Cada vez que escribas una función recursiva, en el fondo estás construyendo una demostración por inducción,
y cada vez que quieras probar que un algoritmo es correcto para *cualquier* tamaño de entrada, la inducción
es tu herramienta. Es una de las ideas que más vas a reutilizar a lo largo de la carrera. 🎓